# EX — Generative AI Fundamentals Real-World Exercises

Hands-on with tokenization, a toy self-attention computation (NumPy, no framework
needed), and autoregressive next-token sampling intuition.


## 1. Tokenization — text isn't words, it's tokens
**Pointer:** token count != word count. This affects cost and context-window budgeting.

In [ ]:
# A tiny illustrative (not production-grade) whitespace+subword-ish tokenizer
import re

def naive_tokenize(text):
    # split on spaces and punctuation, roughly mimicking subword-ish splitting for demo purposes
    return re.findall(r"\w+|[^\w\s]", text)

text = "Tokenization isn't the same as counting words!"
tokens = naive_tokenize(text)
print(tokens)
print("word count (naive):", len(text.split()))
print("token count (naive):", len(tokens))


### TODO 1
Write a function `estimate_cost(text, price_per_1k_tokens=0.002)` using the naive tokenizer above, and use it to compare the cost of a short vs. a long prompt.

In [ ]:
# TODO
def estimate_cost(text, price_per_1k_tokens=0.002):
    pass

short_prompt = "Summarize this."
long_prompt = "Summarize this document in detail, covering every section, sub-point, and nuance thoroughly. " * 5
print(estimate_cost(short_prompt), estimate_cost(long_prompt))


<details><summary>Solution</summary>

```python
def estimate_cost(text, price_per_1k_tokens=0.002):
    n_tokens = len(naive_tokenize(text))
    return (n_tokens/1000) * price_per_1k_tokens
```
</details>

## 2. Self-Attention — a toy computation from scratch
**Pointer:** attention = a learned weighting of how much each token should attend to every other token, based on similarity between 'query' and 'key' vectors.

In [ ]:
import numpy as np
np.random.seed(0)

seq_len, d = 4, 8   # 4 tokens, embedding dim 8
X = np.random.randn(seq_len, d)  # pretend these are token embeddings

Wq = np.random.randn(d, d) * 0.1
Wk = np.random.randn(d, d) * 0.1
Wv = np.random.randn(d, d) * 0.1

Q = X @ Wq
K = X @ Wk
V = X @ Wv

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

scores = Q @ K.T / np.sqrt(d)     # (seq_len, seq_len) similarity between every pair of tokens
attn_weights = softmax(scores)     # each row sums to 1 -- "how much token i attends to token j"
output = attn_weights @ V

print("Attention weights (rows sum to 1):\n", np.round(attn_weights, 2))
print("Output shape:", output.shape)


### TODO 2
Modify token 0's embedding to be very similar to token 2's embedding (e.g. `X[0] = X[2] + small noise`), recompute attention, and observe: does token 0 now attend more strongly to token 2?

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
X2 = X.copy()
X2[0] = X2[2] + np.random.randn(d) * 0.01
Q2, K2, V2 = X2 @ Wq, X2 @ Wk, X2 @ Wv
scores2 = Q2 @ K2.T / np.sqrt(d)
attn2 = softmax(scores2)
print(np.round(attn2, 2))
# Row 0 should now show a notably higher weight on column 2 than before.
```
</details>


## 3. Autoregressive Generation — next-token sampling
**Pointer:** temperature controls how 'peaked' vs. 'flat' the probability distribution is when sampling the next token.

In [ ]:
def softmax_with_temp(logits, temperature=1.0):
    logits = np.array(logits) / max(temperature, 1e-6)
    return softmax(logits)

logits = [2.0, 1.0, 0.1, -1.0]  # pretend these are next-token scores for 4 candidate words
for temp in [0.2, 1.0, 2.0]:
    probs = softmax_with_temp(logits, temp)
    print(f"temperature={temp}: probs={np.round(probs,3)}")


### TODO 3
Explain in one sentence (as a markdown/comment) why low temperature makes output more deterministic and high temperature makes it more random, based on what you observe in the printed probabilities above.

In [ ]:
# TODO: write your one-sentence explanation as a comment


<details><summary>Discussion</summary>

Low temperature sharpens the distribution toward the highest-scoring token (more deterministic, repetitive), while high temperature flattens the distribution so lower-scoring tokens become nearly as likely to be picked (more random/creative, but also more error-prone).
</details>

## Key Takeaways
- Tokens, not words, are the real unit of cost/context — always think in tokens.
- Self-attention is a learned similarity-weighted average across tokens — this is what lets a Transformer use context.
- Temperature reshapes the next-token probability distribution; low = deterministic, high = diverse/risky.
- These same primitives (tokenize -> attend -> sample) underlie every modern LLM you call via an API.
